# Periodogram Covariance — Gaussian Formula Validation

## 목적

$$\mathrm{Cov}(I_{\mathbf{j}}, I_{\mathbf{k}}) = |S_{\mathbf{jk}}|^2 + |P_{\mathbf{jk}}|^2$$

이 식이 수학적으로 맞는지 시뮬레이션으로 직접 검증한다.

**방법**:
1. 알려진 covariance 함수 $C(\mathbf{h})$ 로 Gaussian field 를 직접 $M\times N$ 그리드에 생성
2. True spectral density $f_{\mathrm{true}}(\boldsymbol{\omega})$ 를 analytic하게 계산
3. $\widehat{\mathrm{Cov}}_{\mathrm{gauss}}$ 를 **true $f$** 로 계산 (plug-in bias 없음)
4. 경험적 $\widehat{\mathrm{Cov}}_{\mathrm{emp}}$ 와 비교

**High-res 불필요**: 수식 검증이 목적이므로 high-res subsampling 필요 없음.  
True $f$ 를 쓰면 plug-in bias 없이 식 자체의 정확성을 직접 본다.

**예상 결과**: $N_{\mathrm{snap}} \to \infty$ 에서 $\widehat{\mathrm{Cov}}_{\mathrm{emp}} \to \mathrm{Cov}_{\mathrm{gauss}}(f_{\mathrm{true}})$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fftn, ifftn
from matplotlib.patches import Patch

M_grid, N_grid = 114, 159
DLAT, DLON     = 0.044, 0.063
N_SNAP         = 500
RNG_SEED       = 42

H_data  = np.outer(np.hanning(M_grid), np.hanning(N_grid))
H_data /= np.sqrt(np.mean(H_data**2))
F_H2_data = fftn(H_data**2)
H_dft     = fftn(H_data)
MN  = M_grid * N_grid
MN2 = float(MN)**2

# KEY: pairs must differ in ONLY ONE axis to have non-zero S_jk overlap.
# The outer-product Hann taper H[j-m] ~ W1[j1-m1]*W2[j2-m2], which is
# negligible when |j1-m1|>1 OR |j2-m2|>1.  For S_jk = sum_m f*H_j*H_kc to
# be non-zero we need |j1-k1|<=2 AND |j2-k2|<=2.  But power falls fast:
#   sep=(0,1)  -> corr~0.45   (1D-like leakage in lon axis)
#   sep=(1,0)  -> corr~0.45   (1D-like leakage in lat axis)
#   sep=(1,2)  -> corr~0.01   (product of two small overlaps)
# The original Low/Mid/High bands all had sep=(1,2) or larger in BOTH dims
# -> trivially zero -> no non-trivial formula validation.

FREQ_BANDS = {
    # 4 adjacent lon-freqs at fixed lat=5  (leakage coupling visible)
    'NearLon': [(5, 8), (5, 9),  (5, 10), (5, 11)],
    # 4 adjacent lat-freqs at fixed lon=20  (leakage coupling visible)
    'NearLat': [(8, 20), (9, 20), (10, 20), (11, 20)],
    # well-separated (no leakage coupling expected)
    'Far':     [(20, 40), (35, 60), (50, 75)],
}
FREQ_ALL, BAND_LABELS = [], []
for band, freqs in FREQ_BANDS.items():
    for j in freqs:
        FREQ_ALL.append(j); BAND_LABELS.append(band)
K = len(FREQ_ALL)

sig_thresh = 2 / np.sqrt(N_SNAP)
print(f'Grid: {M_grid}x{N_grid}  |  N_SNAP={N_SNAP}  |  K={K}  |  2/sqrt(N)={sig_thresh:.4f}')
print(f'Bands: {list(FREQ_BANDS.keys())}')
print(f'Expected: NearLon/NearLat adjacent pairs -> corr~0.45; Far -> corr~0')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# True covariance + spectral density on M×N grid
#
# Purely spatial (t=0 slice):
#   C(h_lat, h_lon) = (phi1/phi2) * exp(-phi2 * sqrt(phi3*h_lat^2 + h_lon^2))
#
# True spectral density (circulant approx on finite grid):
#   f_true = FFT(C)   (real, non-negative)
# ─────────────────────────────────────────────────────────────────────────────
true_dict = {
    'sigmasq': 13.059, 'range_lat': 0.154, 'range_lon': 0.195, 'nugget': 0.247,
}
phi2 = 1.0 / true_dict['range_lon']
phi1 = true_dict['sigmasq'] * phi2
phi3 = (true_dict['range_lon'] / true_dict['range_lat'])**2

i_idx = np.arange(M_grid); i_idx[M_grid//2:] -= M_grid
j_idx = np.arange(N_grid); j_idx[N_grid//2:] -= N_grid
H_lat, H_lon = np.meshgrid(i_idx * DLAT, j_idx * DLON, indexing='ij')

dist      = np.sqrt(phi3 * H_lat**2 + H_lon**2)
C_spatial = (phi1 / phi2) * np.exp(-phi2 * dist)
C_spatial[0, 0] += true_dict['nugget']

f_true = np.real(fftn(C_spatial))
f_true = np.clip(f_true, 0, None)

print(f"C(0,0) = {C_spatial[0,0]:.4f}  (= sigmasq+nugget = {true_dict['sigmasq']+true_dict['nugget']:.4f})")
print(f"f_true: min={f_true.min():.4f}  max={f_true.max():.4f}")
print(f"\nf_true at selected frequencies:")
for q, (j, band) in enumerate(zip(FREQ_ALL, BAND_LABELS)):
    print(f"  {band:5s} ({j[0]:3d},{j[1]:3d})  f_true={f_true[j[0],j[1]]:.4f}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Gaussian field generation: X = IFFT(sqrt(f_true) * Z),  Z ~ CN(0,1)
# ─────────────────────────────────────────────────────────────────────────────
rng    = np.random.default_rng(RNG_SEED)
scale  = np.sqrt(MN)
sqrt_f = np.sqrt(f_true)

J_tap  = np.zeros((N_SNAP, K), dtype=complex)
J_raw  = np.zeros((N_SNAP, K), dtype=complex)
f_plug = np.zeros((M_grid, N_grid), dtype=float)

print(f'Generating {N_SNAP} Gaussian snapshots ...')
for n in range(N_SNAP):
    Z = (rng.standard_normal((M_grid, N_grid))
         + 1j * rng.standard_normal((M_grid, N_grid))) / np.sqrt(2)
    X = np.real(ifftn(sqrt_f * Z)) * scale * np.sqrt(2)  # real() halves variance

    g = X - X.mean()
    F_tap = fftn(g * H_data)
    F_raw = fftn(g)
    f_plug += np.abs(F_tap)**2 / MN
    for q, (j1, j2) in enumerate(FREQ_ALL):
        J_tap[n, q] = F_tap[j1, j2] / scale
        J_raw[n, q] = F_raw[j1, j2] / scale

f_plug /= N_SNAP
I_tap      = np.abs(J_tap)**2
I_raw      = np.abs(J_raw)**2
f_hat_tap  = I_tap.mean(axis=0)
f_hat_raw  = I_raw.mean(axis=0)

def emp_cov(I):
    Ic = I - I.mean(axis=0)
    return (Ic.T @ Ic) / I.shape[0]

Cov_emp_tap = emp_cov(I_tap)
Cov_emp_raw = emp_cov(I_raw)

print('Done.')
print(f"\n{'q':>3}  {'Band':>5}  {'freq':>12}  {'f_true':>10}  {'f_hat':>10}  {'ratio':>8}")
print('-'*56)
for q, (j, band) in enumerate(zip(FREQ_ALL, BAND_LABELS)):
    ft = f_true[j[0], j[1]]
    print(f"  {q:2d}  {band:5s}  ({j[0]:3d},{j[1]:3d})  {ft:10.4f}  {f_hat_tap[q]:10.4f}  {f_hat_tap[q]/ft:8.3f}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Gaussian baselines
#   A. True f     — formula validation (no bias)
#   B. Plug-in f̂  — same formula with empirical spectrum
#   C. Smooth     — Priestley approx with plug-in
# ─────────────────────────────────────────────────────────────────────────────
m1 = np.arange(M_grid)[:, None]
m2 = np.arange(N_grid)[None, :]

def exact_cov_gauss(f_spectrum):
    C = np.zeros((K, K))
    for i, (j1, j2) in enumerate(FREQ_ALL):
        H_j = H_dft[(j1-m1)%M_grid, (j2-m2)%N_grid]
        for l, (k1, k2) in enumerate(FREQ_ALL):
            H_kc = np.conj(H_dft[(k1-m1)%M_grid, (k2-m2)%N_grid])
            H_kp = H_dft[(k1+m1)%M_grid, (k2+m2)%N_grid]
            S_jk = np.sum(f_spectrum * H_j * H_kc) / MN2
            P_jk = np.sum(f_spectrum * H_j * H_kp) / MN2
            C[i, l] = abs(S_jk)**2 + abs(P_jk)**2
    return C

print('Computing baselines ...')
Cov_gauss_true   = exact_cov_gauss(f_true)
print('  A. Exact(true f) done')
Cov_gauss_plugin = exact_cov_gauss(f_plug)
print('  B. Exact(plug-in) done')

Cov_gauss_smooth = np.zeros((K, K))
for i, (j1, j2) in enumerate(FREQ_ALL):
    for l, (k1, k2) in enumerate(FREQ_ALL):
        dm = ((j1-k1)%M_grid, (j2-k2)%N_grid)
        dp = ((j1+k1)%M_grid, (j2+k2)%N_grid)
        Cov_gauss_smooth[i,l] = (f_hat_tap[i]*f_hat_tap[l]
            * (abs(F_H2_data[dm])**2+abs(F_H2_data[dp])**2) / MN2)
print('  C. Smooth(Priestley) done')

print(f"\n{'q':>3}  {'Band':>5}  {'freq':>12}  "
      f"{'Var_emp':>10}  {'Gauss(true)':>13}  {'Gauss(plug)':>13}  {'Smooth':>10}")
print('-'*72)
for q, (j, band) in enumerate(zip(FREQ_ALL, BAND_LABELS)):
    print(f"  {q:2d}  {band:5s}  ({j[0]:3d},{j[1]:3d})  "
          f"{Cov_emp_tap[q,q]:10.4f}  {Cov_gauss_true[q,q]:13.4f}  "
          f"{Cov_gauss_plugin[q,q]:13.4f}  {Cov_gauss_smooth[q,q]:10.4f}")

In [ ]:
band_colors_ = {'NearLon': '#2166ac', 'NearLat': '#d6604d', 'Far': '#1a9641'}
tick_labels_ = [f'({j[0]},{j[1]})' for j in FREQ_ALL]
tick_colors_ = [band_colors_[b] for b in BAND_LABELS]

def to_corr(cov):
    std = np.sqrt(np.clip(np.diag(cov), 1e-30, None))
    return cov / np.outer(std, std)

def draw_mat(ax, mat, cmap, vmin, vmax, title):
    im = ax.imshow(mat, cmap=cmap, vmin=vmin, vmax=vmax, aspect='auto')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
    ax.set_title(title, fontsize=8.5, fontweight='bold')
    ax.set_xticks(range(K)); ax.set_yticks(range(K))
    ax.set_xticklabels(tick_labels_, rotation=45, ha='right', fontsize=6)
    ax.set_yticklabels(tick_labels_, fontsize=6)
    for q, tc in enumerate(tick_colors_):
        ax.get_xticklabels()[q].set_color(tc)
        ax.get_yticklabels()[q].set_color(tc)
    # group separators: NearLon=4, NearLat=4, Far=3
    for sep in [3.5, 7.5]:
        ax.axhline(sep, color='gray', lw=0.8, ls='--')
        ax.axvline(sep, color='gray', lw=0.8, ls='--')
    for i in range(K):
        for l in range(K):
            ax.text(l, i, f'{mat[i,l]:.2f}', ha='center', va='center',
                    fontsize=5, color='white' if abs(mat[i,l])>0.5 else 'black')

Cemp  = to_corr(Cov_emp_tap)
Ctrue = to_corr(Cov_gauss_true)
Cplug = to_corr(Cov_gauss_plugin)
Csmth = to_corr(Cov_gauss_smooth)

rows = [
    (Cemp, Ctrue, 'Exact(true f) — FORMULA CHECK',   'Gauss_true'),
    (Cemp, Cplug, 'Exact(plug-in f) — PLUG-IN BIAS', 'Gauss_plugin'),
    (Cemp, Csmth, 'Smooth approx (Priestley)',        'Gauss_smooth'),
]

fig, axes = plt.subplots(3, 3, figsize=(21, 21))
fig.suptitle(
    f"GAUSSIAN SIMULATION — Formula Validation  |  N={N_SNAP} snapshots\n"
    f"NearLon/NearLat: adjacent pairs (corr~0.45 expected)  |  Far: well-separated (corr~0)",
    fontsize=12, fontweight='bold')

for row_idx, (Cemp_r, Cgauss, row_tag, gauss_name) in enumerate(rows):
    res  = Cemp_r - Cgauss
    rlim = max(0.05, np.abs(res).max())
    draw_mat(axes[row_idx,0], Cemp_r, 'RdBu_r', -1, 1, 'Empirical Corr  [SIM]')
    draw_mat(axes[row_idx,1], Cgauss, 'RdBu_r', -1, 1,
             f'Gaussian Baseline\n[{row_tag}]')
    draw_mat(axes[row_idx,2], res,    'PiYG', -rlim, rlim,
             f'Residual  [{gauss_name}]\nmax={rlim:.3f}  (approx 0 if formula correct)')

handles = [Patch(color=c, label=b) for b, c in band_colors_.items()]
axes[0,0].legend(handles=handles, loc='lower right', fontsize=8, title='Group')
plt.tight_layout()
plt.savefig('/tmp/cov_sim_formula_check.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: /tmp/cov_sim_formula_check.png')


In [ ]:
BANDS_IDX  = {b: [i for i,bl in enumerate(BAND_LABELS) if bl==b]
              for b in ['NearLon', 'NearLat', 'Far']}
BAND_PAIRS = [('NearLon','NearLon'),('NearLat','NearLat'),('NearLon','NearLat'),
              ('NearLon','Far'),('NearLat','Far'),('Far','Far')]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(
    f'SIM Off-diagonal |Corr|  (N={N_SNAP})\n'
    f'Blue=|Corr_emp|  Orange=|Gauss(true f)|  Red=|Gauss(plug-in)|  '
    f'Dashed=2/sqrt(N)={sig_thresh:.3f}\n'
    f'NearLon/NearLat within-group: expect ~0.45 (non-trivial validation!)',
    fontsize=10, fontweight='bold')
axes = axes.ravel()
summary = []

for ax_idx, (b1, b2) in enumerate(BAND_PAIRS):
    ax = axes[ax_idx]
    pe, pt, pp, lbls = [], [], [], []
    for i in BANDS_IDX[b1]:
        for l in BANDS_IDX[b2]:
            if b1==b2 and i==l: continue
            pe.append(abs(Cemp[i,l]))
            pt.append(abs(Ctrue[i,l]))
            pp.append(abs(Cplug[i,l]))
            lbls.append(f'({FREQ_ALL[i][0]},{FREQ_ALL[i][1]})\nx({FREQ_ALL[l][0]},{FREQ_ALL[l][1]})')
    x = np.arange(len(pe)); w = 0.25
    col = band_colors_.get(b1, '#555555')
    ax.bar(x-w, pe, w, color=col,          alpha=0.85, edgecolor='w', label='|Corr_emp|')
    ax.bar(x,   pt, w, color='darkorange', alpha=0.70, edgecolor='w', label='|Gauss(true f)|')
    ax.bar(x+w, pp, w, color='firebrick',  alpha=0.55, edgecolor='w', label='|Gauss(plug-in)|')
    ax.axhline(sig_thresh, color='k', ls='--', lw=1.0, label='2/sqrt(N)')
    ax.set_title(f'{b1} x {b2}', fontsize=10, fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels(lbls, fontsize=6.5)
    ax.set_ylabel('|Corr|', fontsize=9)
    ax.set_ylim(0, max(0.05, max(pe+pt+pp+[sig_thresh])*1.3) if pe else 0.1)
    ax.legend(fontsize=7)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, ls='--', alpha=0.35); ax.set_axisbelow(True)
    if pe:
        summary.append((b1, b2, max(pe), max(pt), sum(v>sig_thresh for v in pe), len(pe)))

plt.tight_layout()
plt.savefig('/tmp/offdiag_sim_formula.png', dpi=130, bbox_inches='tight')
plt.show()

print(f'\nFormula check: Corr_emp vs Gauss(true f)')
print(f"  NearLon/NearLat within-group: emp should match true (~0.45)")
print(f"  Far/cross-group: both should be ~0")
print(f"{'Pair':>22}  {'max|emp|':>10}  {'max|true|':>11}  {'n_sig/n':>10}")
print('-'*60)
for b1_,b2_,me,mt,ns,np_ in summary:
    match = 'MATCH' if abs(me-mt) < sig_thresh else 'DIFF'
    print(f'  {b1_:10s}x{b2_:10s}  {me:10.4f}  {mt:11.4f}  {ns}/{np_}  {match}')


In [ ]:
var_ratio    = np.array([Cov_emp_tap[q,q] / Cov_gauss_true[q,q] for q in range(K)])
offdiag_emp  = [abs(Cemp[i,l])             for i in range(K) for l in range(K) if i!=l]
offdiag_true = [abs(Ctrue[i,l])            for i in range(K) for l in range(K) if i!=l]
offdiag_diff = [abs(Cemp[i,l]-Ctrue[i,l]) for i in range(K) for l in range(K) if i!=l]
n_sig_diff   = sum(v > sig_thresh for v in offdiag_diff)
n_total      = len(offdiag_emp)

print('='*65)
print('  FORMULA VALIDATION SUMMARY')
print('='*65)
print(f'  N_SNAP = {N_SNAP}  |  2/sqrt(N) = {sig_thresh:.4f}')
print()
print(f'  [Diagonal]  Var_emp / Cov_gauss(true f):')
print(f'  mean={var_ratio.mean():.3f}  min={var_ratio.min():.3f}  max={var_ratio.max():.3f}')
print(f'  (should be approx 1.0 as N_SNAP -> inf)')
print()
print(f'  [Off-diagonal]')
print(f'  max |Corr_emp|              = {max(offdiag_emp):.4f}')
print(f'  max |Corr_gauss(true f)|    = {max(offdiag_true):.4f}  (approx 0 by formula)')
print(f'  max |Corr_emp - Corr_true|  = {max(offdiag_diff):.4f}')
print(f'  n(|emp-true| > 2/sqrt(N))   = {n_sig_diff}/{n_total}  ({100*n_sig_diff/n_total:.1f}%)')
print()
if n_sig_diff / n_total < 0.10:
    print('  PASS: Cov_emp approx Cov_gauss(true f)')
    print('  -> formula validated; GEMS excess coupling is non-Gaussian')
else:
    print('  WARNING: significant residuals remain -- check formula')
print('='*65)